# 05 — Product Network

**Project:** SERSA Product Demand Relationship Analysis  
**Author:** Cesar Enrique Banda Martinez  
**Date:** 2026  

---

## Context

This notebook is the final phase of the satellite project.  
It synthesizes the findings from notebooks 02–04 into a **product demand network** — a graph where:

- Each **node** is a SKU.
- Each **edge** represents a strong demand relationship (growth-rate correlation r ≥ 0.75).
- **Edge weight** encodes correlation strength.
- **Edge color** distinguishes contemporaneous relationships (lag = 0) from lead-lag relationships (lag ≠ 0).

### Why a network visualization?
A correlation matrix (notebooks 02–03) shows all pairwise relationships simultaneously, but becomes unreadable at 86 SKUs. A network graph filters to meaningful edges only and reveals **cluster structure** — groups of SKUs that behave as demand communities — which is invisible in a heatmap.

### Summary of findings carried into this notebook

| Layer | Finding |
|-------|---------|
| Raw correlation (nb 02) | 298 pairs with r ≥ 0.75 — inflated by shared growth trend |
| Growth-rate correlation (nb 03) | 104 genuine pairs survive detrending |
| Lead-lag analysis (nb 04) | 96 pairs contemporaneous; 8 pairs with lag ≠ 0, of which 2 are analytically strong (r > 0.97): `UVS4040 → UVSVP401` and `ME6632XL → UVSVP401` at lag −2 months |

### Goals of this notebook
1. Build the product demand network from growth-rate correlations.
2. Detect communities (clusters) using a graph algorithm.
3. Visualize the full network with edge weights and lead-lag highlights.
4. Produce a focused subgraph of the strongest relationships.
5. Export network data for potential Power BI integration.

---

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

---
## 2. Configuration

In [ ]:
DATA_DIR       = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_TABLES  = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_FIGURES = os.path.join(os.getcwd(), "..", "outputs", "figures")

CORR_THRESHOLD     = 0.75   # minimum edge weight
STRONG_THRESHOLD   = 0.90   # threshold for focused subgraph

print(f"Data dir:       {os.path.normpath(DATA_DIR)}")
print(f"Output tables:  {os.path.normpath(OUTPUT_TABLES)}")
print(f"Output figures: {os.path.normpath(OUTPUT_FIGURES)}")

---
## 3. Load Data

In [ ]:
# Growth-rate top pairs (edges)
top_pairs = pd.read_csv(
    os.path.join(DATA_DIR, "03_top_positive_pairs_growth.csv"),
    decimal=","
)

# Lead-lag dominant lag per pair
dominant_lag = pd.read_csv(
    os.path.join(DATA_DIR, "04_dominant_lag_per_pair.csv"),
    decimal=","
)

# Merge lag info into pairs
edges = top_pairs.merge(
    dominant_lag[["SKU_A", "SKU_B", "dominant_lag"]],
    on=["SKU_A", "SKU_B"],
    how="left"
)
edges["dominant_lag"] = edges["dominant_lag"].fillna(0).astype(int)

print(f"Edges loaded: {len(edges)}")
print()
print(edges.head(8).to_string(index=False))

---
## 4. Build Network

In [ ]:
G = nx.Graph()

for _, row in edges.iterrows():
    G.add_edge(
        row["SKU_A"],
        row["SKU_B"],
        weight=row["Pearson_r"],
        lag=row["dominant_lag"]
    )

print(f"Nodes : {G.number_of_nodes()}")
print(f"Edges : {G.number_of_edges()}")
print(f"Connected components: {nx.number_connected_components(G)}")

---
## 5. Community Detection

We use the **Louvain-style greedy modularity** algorithm available in NetworkX to detect communities — groups of SKUs more densely connected to each other than to the rest of the network.

This is an unsupervised method: it finds natural clusters without needing to specify the number of groups.

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G, weight="weight"))
communities = sorted(communities, key=len, reverse=True)

print(f"Communities detected: {len(communities)}")
print()
for i, comm in enumerate(communities):
    print(f"  Community {i+1} ({len(comm)} SKUs): {sorted(comm)}")

---
## 6. Assign Node Attributes

In [ ]:
# Community membership
community_map = {}
for i, comm in enumerate(communities):
    for sku in comm:
        community_map[sku] = i

nx.set_node_attributes(G, community_map, "community")

# Node degree (number of strong connections)
degree_map = dict(G.degree())
nx.set_node_attributes(G, degree_map, "degree")

print("Top 10 nodes by degree (most connected SKUs):")
top_degree = sorted(degree_map.items(), key=lambda x: x[1], reverse=True)[:10]
for sku, deg in top_degree:
    print(f"  {sku:<25} degree = {deg}")

---
## 7. Full Network Visualization

In [ ]:
# Color palette per community
COMMUNITY_COLORS = [
    "#2C7BB6", "#D7191C", "#1A9641", "#FDAE61",
    "#7B2D8B", "#F46D43", "#006837", "#ABD9E9",
    "#A6D96A", "#878787"
]

def get_node_color(node):
    comm = community_map.get(node, 0)
    return COMMUNITY_COLORS[comm % len(COMMUNITY_COLORS)]

# Layout
pos = nx.spring_layout(G, weight="weight", seed=42, k=2.5)

node_colors  = [get_node_color(n) for n in G.nodes()]
node_sizes   = [200 + degree_map[n] * 80 for n in G.nodes()]
edge_weights = [G[u][v]["weight"] for u, v in G.edges()]
edge_lags    = [G[u][v]["lag"]    for u, v in G.edges()]
edge_colors  = ["#FF6600" if lag != 0 else "#AAAAAA" for lag in edge_lags]
edge_widths  = [0.5 + w * 2 for w in edge_weights]

fig, ax = plt.subplots(figsize=(20, 16))

nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_colors,
    width=edge_widths,
    alpha=0.6,
    ax=ax
)

nx.draw_networkx_nodes(
    G, pos,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.92,
    ax=ax
)

nx.draw_networkx_labels(
    G, pos,
    font_size=5.5,
    font_color="black",
    ax=ax
)

# Legend — communities
legend_patches = []
for i, comm in enumerate(communities):
    if len(comm) >= 2:
        label = f"Community {i+1} ({len(comm)} SKUs)"
        legend_patches.append(mpatches.Patch(color=COMMUNITY_COLORS[i % len(COMMUNITY_COLORS)], label=label))

legend_patches.append(mpatches.Patch(color="#AAAAAA", label="Contemporaneous edge (lag = 0)"))
legend_patches.append(mpatches.Patch(color="#FF6600", label="Lead-lag edge (lag \u2260 0)"))

ax.legend(handles=legend_patches, loc="upper left", fontsize=8, framealpha=0.9)
ax.set_title(
    f"Product Demand Network\n{G.number_of_nodes()} SKUs, {G.number_of_edges()} edges (growth-rate r \u2265 {CORR_THRESHOLD})",
    fontsize=15, fontweight="bold"
)
ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "05_product_network_full.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 05_product_network_full.png")

---
## 8. Focused Subgraph — Strongest Relationships

A subgraph filtered to edges with r ≥ 0.90 highlights only the most robust demand relationships,  
making the core cluster structure easier to communicate.

In [ ]:
strong_edges = [(u, v) for u, v, d in G.edges(data=True) if d["weight"] >= STRONG_THRESHOLD]
G_strong = G.edge_subgraph(strong_edges).copy()

print(f"Strong subgraph: {G_strong.number_of_nodes()} nodes, {G_strong.number_of_edges()} edges (r >= {STRONG_THRESHOLD})")

degree_strong = dict(G_strong.degree())
pos_strong = nx.spring_layout(G_strong, weight="weight", seed=42, k=3.0)

node_colors_s  = [get_node_color(n) for n in G_strong.nodes()]
node_sizes_s   = [300 + degree_strong[n] * 100 for n in G_strong.nodes()]
edge_weights_s = [G_strong[u][v]["weight"] for u, v in G_strong.edges()]
edge_lags_s    = [G_strong[u][v]["lag"]    for u, v in G_strong.edges()]
edge_colors_s  = ["#FF6600" if lag != 0 else "#AAAAAA" for lag in edge_lags_s]
edge_widths_s  = [0.5 + w * 3 for w in edge_weights_s]

fig, ax = plt.subplots(figsize=(16, 12))

nx.draw_networkx_edges(
    G_strong, pos_strong,
    edge_color=edge_colors_s,
    width=edge_widths_s,
    alpha=0.7,
    ax=ax
)

nx.draw_networkx_nodes(
    G_strong, pos_strong,
    node_color=node_colors_s,
    node_size=node_sizes_s,
    alpha=0.92,
    ax=ax
)

nx.draw_networkx_labels(
    G_strong, pos_strong,
    font_size=7,
    font_color="black",
    ax=ax
)

ax.legend(handles=legend_patches, loc="upper left", fontsize=9, framealpha=0.9)
ax.set_title(
    f"Product Demand Network — Strong Relationships Only\n{G_strong.number_of_nodes()} SKUs, {G_strong.number_of_edges()} edges (growth-rate r \u2265 {STRONG_THRESHOLD})",
    fontsize=14, fontweight="bold"
)
ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "05_product_network_strong.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 05_product_network_strong.png")

---
## 9. Network Summary Statistics

In [ ]:
print("=" * 50)
print("NETWORK SUMMARY")
print("=" * 50)
print(f"  Nodes (SKUs)        : {G.number_of_nodes()}")
print(f"  Edges               : {G.number_of_edges()}")
print(f"  Communities         : {len(communities)}")
print(f"  Connected components: {nx.number_connected_components(G)}")
print(f"  Avg clustering coef : {nx.average_clustering(G, weight='weight'):.3f}")
print()

print("Community sizes:")
for i, comm in enumerate(communities):
    print(f"  Community {i+1}: {len(comm)} SKUs")

print()
print("Lead-lag edges:")
ll_edges = [(u, v, d) for u, v, d in G.edges(data=True) if d["lag"] != 0]
for u, v, d in sorted(ll_edges, key=lambda x: abs(x[2]["weight"]), reverse=True):
    direction = f"{u} leads {v}" if d["lag"] < 0 else f"{v} leads {u}"
    print(f"  {u:<25} \u2194 {v:<25} lag={d['lag']:+d}  r={d['weight']:.4f}  ({direction})")

---
## 10. Export

In [ ]:
# Edge list for Power BI / external tools
edge_records = []
for u, v, d in G.edges(data=True):
    edge_records.append({
        "SKU_A"        : u,
        "SKU_B"        : v,
        "Pearson_r"    : round(d["weight"], 4),
        "dominant_lag" : d["lag"],
        "is_lead_lag"  : int(d["lag"] != 0),
        "community_A"  : community_map.get(u, -1),
        "community_B"  : community_map.get(v, -1),
    })

edge_df = pd.DataFrame(edge_records)

# Node list
node_records = []
for n in G.nodes():
    node_records.append({
        "SKU"       : n,
        "community" : community_map.get(n, -1),
        "degree"    : degree_map[n],
    })

node_df = pd.DataFrame(node_records).sort_values("degree", ascending=False).reset_index(drop=True)

edge_df.to_csv(os.path.join(OUTPUT_TABLES, "05_network_edges.csv"), index=False, decimal=",")
node_df.to_csv(os.path.join(OUTPUT_TABLES, "05_network_nodes.csv"), index=False, decimal=",")

print("Export complete.")
print(f"  05_network_edges.csv  -> {len(edge_df)} edges")
print(f"  05_network_nodes.csv  -> {len(node_df)} nodes")

---
## 11. Project Summary

This notebook closes the *SERSA Product Demand Relationship Analysis* project.  
Below is a consolidated summary of all findings across the five notebooks.

---

### Layer 1 — Raw Correlation (notebook 02)
- 298 SKU pairs with Pearson r ≥ 0.75 on raw monthly transaction counts.
- Dominated by shared business growth trend (294% revenue increase 2022–2025).

### Layer 2 — Growth-Rate Correlation (notebook 03)
- After detrending via `pct_change()`, **104 genuine pairs** survive (65% of raw pairs were spurious).
- **7 negative pairs** emerge — invisible in raw correlations — suggesting potential substitution dynamics.
- Size-variant clusters (SF1340 family, AN11840 family, SFPUG111 family) remain strongly correlated: operators dispense all sizes together, making variants behave as a single demand unit.

### Layer 3 — Lead-Lag Analysis (notebook 04)
- **96 of 104 pairs are purely contemporaneous** — demand co-movement occurs within the same month.
- **8 pairs show lead-lag structure**; 2 are analytically robust (r > 0.97 at lag ≠ 0):
  - `UVS4040` and `ME6632XL` each lead `UVSVP401` by approximately 2 months.
  - This is the only confirmed predictive signal in the dataset.

### Layer 4 — Product Network (notebook 05)
- Network built from 104 genuine edges; communities detected via greedy modularity.
- Lead-lag edges highlighted as forward signals for demand planning.

---

**End of project.**